# 00 — Rebuild Dataset & Lane Mask Cache

**Purpose:** Download/mount BDD100K data, parse poly2d lane annotations,
render binary lane masks to disk, and verify the dataset pipeline loads correctly.

**Outputs (saved to Drive):**
- `bdd100k/lane_masks/train/*.png` — binary lane masks (640×640)
- `bdd100k/lane_masks/val/*.png` — binary lane masks (640×640)

**Prerequisites:** BDD100K images + lane JSON labels accessible on Drive or Colab disk.

In [ ]:
# ── Mount Drive & install deps ──
from google.colab import drive
drive.mount('/content/drive')

!pip install -q yacs tqdm opencv-python-headless

In [ ]:
# ── Clone or symlink the repo ──
import os, sys

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/EcoCAR-Perception-Pipeline-YOLO26-BDD100K/yolop_vehicle_lane'
os.chdir(REPO_ROOT)
sys.path.insert(0, REPO_ROOT)
print(f'Working dir: {os.getcwd()}')

In [ ]:
# ── Configuration ──
# Adjust these paths to match your Drive layout
BDD_IMAGES = '/content/bdd100k/images/100k'        # {train,val}/*.jpg
BDD_LABELS = '/content/bdd100k/labels/100k'         # {train,val}/*.json
BDD_LANE_JSON_TRAIN = '/content/bdd100k/labels/lane/polygons/lane_train.json'
BDD_LANE_JSON_VAL   = '/content/bdd100k/labels/lane/polygons/lane_val.json'
LANE_MASK_OUT = '/content/bdd100k/lane_masks'       # output dir

MASK_W, MASK_H = 1280, 720   # render at original resolution, resize later in dataset
LINE_THICKNESS = 8

In [ ]:
# ── Parse lane labels and render masks ──
from lib.utils.lane_targets import extract_lane_labels_any
from lib.utils.lane_render import convert_bdd_lanes_to_masks
from pathlib import Path

for split, json_path in [('train', BDD_LANE_JSON_TRAIN), ('val', BDD_LANE_JSON_VAL)]:
    out_dir = Path(LANE_MASK_OUT) / split
    out_dir.mkdir(parents=True, exist_ok=True)
    
    existing = len(list(out_dir.glob('*.png')))
    print(f'\n=== {split} split ===')
    print(f'  JSON: {json_path}')
    print(f'  Output: {out_dir}')
    print(f'  Existing masks: {existing}')
    
    if existing > 100:
        print(f'  Skipping (already have {existing} masks). Delete to regenerate.')
        continue
    
    convert_bdd_lanes_to_masks(
        json_path=json_path,
        output_dir=str(out_dir),
        img_w=MASK_W,
        img_h=MASK_H,
        line_thickness=LINE_THICKNESS
    )

In [ ]:
# ── Verify: count masks and spot-check ──
import cv2
import matplotlib.pyplot as plt
import numpy as np

for split in ['train', 'val']:
    mask_dir = Path(LANE_MASK_OUT) / split
    masks = sorted(mask_dir.glob('*.png'))
    print(f'{split}: {len(masks)} lane masks')

# Visualize a few samples
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
val_masks = sorted((Path(LANE_MASK_OUT) / 'val').glob('*.png'))[:4]
for i, mask_path in enumerate(val_masks):
    mask = cv2.imread(str(mask_path), 0)
    img_path = Path(BDD_IMAGES) / 'val' / (mask_path.stem + '.jpg')
    img = cv2.imread(str(img_path))
    if img is not None:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        axes[0, i].imshow(img)
        axes[0, i].set_title(mask_path.stem[:20])
    axes[1, i].imshow(mask, cmap='gray')
    axes[0, i].axis('off')
    axes[1, i].axis('off')
plt.suptitle('Sample Images + Lane Masks')
plt.tight_layout()
plt.show()

In [ ]:
# ── Smoke-test dataset pipeline ──
from lib.config import cfg, update_config
from lib.dataset import BddDataset
from torch.utils.data import DataLoader
import argparse

# Override paths for this Colab session
cfg.defrost()
cfg.DATASET.DATAROOT = BDD_IMAGES
cfg.DATASET.LABELROOT = BDD_LABELS
cfg.DATASET.LANEROOT = LANE_MASK_OUT
cfg.freeze()

val_dataset = BddDataset(
    cfg, is_train=False, inputsize=640,
    transform=__import__('torchvision').transforms.ToTensor()
)
print(f'Val dataset: {len(val_dataset)} samples')

val_loader = DataLoader(
    val_dataset, batch_size=4, shuffle=False, num_workers=2,
    collate_fn=val_dataset.collate_fn
)

img, target, paths, shapes = next(iter(val_loader))
det_labels, lane_labels = target
print(f'Image batch shape:  {img.shape}')
print(f'Det labels shape:   {det_labels.shape}')
print(f'Lane labels shape:  {lane_labels.shape}')
print('Dataset pipeline OK!')